# Laboratorio de desarrollo: de un cuaderno a un Hugging Face Space

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un modelo que funciona en un cuaderno todavía no es algo que otra persona pueda usar: hace falta un entorno reproducible, un archivo ejecutable y un lugar donde publicarlo. |
| **Producto** | Un Space de Hugging Face que analiza el sentimiento de transcripciones de audio, armado con ayuda de un asistente de IA y comprobado con checkpoints. |
| **Recursos** | Python 3.12, `uv` y Git. `transformers`, `torch` y `gradio`; el modelo RoBERTuito (450 MB). Una cuenta de Hugging Face para la Parte 8. Sin GPU. |

Tu clasificador de sentimientos ya funciona en un cuaderno y querés que un compañero lo pruebe desde el navegador, sin instalar nada. Eso exige estructurar un entorno de desarrollo, pasar el código del cuaderno a un script y publicarlo.

**¿Qué hace falta para pasar de un cuaderno que funciona a una aplicación que otra persona puede usar, y cómo se trabaja ese camino con ayuda de una IA?**

> **Extensión opcional.** No forma parte del recorrido obligatorio. Es un laboratorio guiado: seguí las partes en orden y construí el Space paso a paso en la Parte 9. Si no tenés `uv`, ver `ANTES_DE_LA_CLASE.md`.

## Cuatro términos que vas a usar

| Término | Qué es |
|---|---|
| **Entorno virtual** | Espacio aislado de tu computadora con sus propios paquetes, para que los proyectos no se pisen entre sí. |
| **Hugging Face Spaces** | Servicio de la nube de Hugging Face que publica gratis aplicaciones web de Machine Learning (por ejemplo, demos de Gradio). |
| **Modularización** | Dividir un programa en partes chicas, independientes y reutilizables: módulos o scripts. |
| **Vibe coding y vibe engineering** | Formas de desarrollar con ayuda de IA generativa: iteración rápida, depuración sistemática con contexto y prototipos funcionales (MVP) que se mejoran de a poco. |

## Parte 0: Configuración del entorno de desarrollo

### Por qué los entornos virtuales son críticos

Antes de desarrollar en local, **siempre** hay que crear un entorno virtual. Esto evita conflictos de dependencias y mantiene tu sistema limpio.

**Problemas sin entorno virtual:**

```
Proyecto A necesita transformers 4.35.0
Proyecto B necesita transformers 4.20.0
→ CONFLICTO: uno de los dos va a fallar
```

**Solución: entornos aislados**

```
.venv del proyecto A/
  └─ transformers 4.35.0

.venv del proyecto B/
  └─ transformers 4.20.0

→ Ambos funcionan sin interferencia
```

### Crear el entorno virtual con uv

En el curso trabajamos con **`uv`**, que crea el entorno y instala paquetes mucho más rápido que `python -m venv` y `pip`.

```bash
# Crear la carpeta del proyecto y entrar
mkdir analizador-sentimientos-habla
cd analizador-sentimientos-habla

# Crear el entorno virtual con Python 3.12 (genera la carpeta .venv)
uv venv --python 3.12

# Activar (macOS o Linux)
source .venv/bin/activate

# Activar (Windows, PowerShell)
.venv\Scripts\activate

# Verificar que estás en el entorno: el prompt cambia y muestra su nombre
(analizador-sentimientos-habla) $
```

`uv pip install` detecta solo la carpeta `.venv` del proyecto; activar el entorno hace falta para correr `python app.py`.

### Instalar dependencias en el entorno

Con el entorno activo:

```bash
# Las librerías, en UN solo comando y entre comillas
uv pip install "transformers>=4.48,<5.0" "torch>=2.6" "gradio>=5.0"

# O desde un requirements.txt
uv pip install -r requirements.txt
```

Las comillas importan: sin ellas, la terminal interpreta `>` como una redirección a un archivo. Y conviene instalar todo junto: instalados por separado, `gradio` puede subir `huggingface-hub` a una versión que `transformers` 4.x no acepta.

### Desactivar el entorno

```bash
deactivate
```

### .gitignore para entornos virtuales

**Nunca** subas el entorno virtual a Git. Creá un `.gitignore`:

```
# Entornos virtuales
.venv/
venv/
env/

# Python
__pycache__/
*.pyc
*.pyo
.DS_Store

# IDEs
.vscode/
.idea/
```

## Parte 1: Hugging Face en profundidad

### Qué es Hugging Face

**Hugging Face** es el «GitHub de la IA»: una plataforma que democratiza el acceso a:

- **Models**: cientos de miles de modelos preentrenados (BERT, GPT, Whisper, etc.).
- **Datasets**: cientos de miles de conjuntos de datos públicos (Common Voice, LibriSpeech).
- **Spaces**: aplicaciones de Machine Learning publicadas gratis.
- **Documentación**: cursos y tutoriales completos.

### Qué es un Space

Un **Space** es una aplicación web alojada gratis que ejecuta:

- **Gradio**: interfaces de ML interactivas (la que usamos hoy).
- **Streamlit**: tableros de datos.
- **Static HTML**: páginas web estáticas.
- **Docker**: contenedores personalizados.

**Ventajas de los Spaces:**

- Alojamiento gratuito.
- Despliegue automático al hacer `git push`.
- GPU disponible (con límites).
- URL pública para compartir.
- Se puede embeber en sitios web.

## Parte 2: Anatomía de un Hugging Face Space

### Estructura de archivos obligatoria

```
mi-space-sentimientos/
├── app.py              # ARCHIVO PRINCIPAL: punto de entrada
├── requirements.txt    # DEPENDENCIAS: paquetes necesarios
├── README.md           # METADATOS: configuración del Space
├── .gitignore          # Archivos a ignorar
└── (opcional)
    ├── utils.py        # Funciones auxiliares
    └── examples/       # Archivos de ejemplo
```

### El archivo app.py: el corazón del Space

```python
"""
app.py - Clasificador de sentimientos para transcripciones de audio.
Este archivo debe llamarse app.py para que Hugging Face lo reconozca.
"""

import gradio as gr
from transformers import pipeline

# 1. CARGAR EL MODELO (se ejecuta UNA vez, al inicio)
print("Cargando modelo RoBERTuito...")
clasificador = pipeline(
    "text-classification",
    model="pysentimiento/robertuito-sentiment-analysis",
)
print("Modelo cargado")


# 2. FUNCION DE PROCESAMIENTO
def analizar_sentimiento(texto: str) -> dict:
    """Analiza el sentimiento de una transcripcion de audio.

    Args:
        texto: Transcripcion del audio.

    Returns:
        Diccionario {etiqueta: puntaje}, el formato que espera gr.Label.
    """
    if not texto.strip():
        raise gr.Error("Escribi o pega una transcripcion.")

    resultado = clasificador(texto)[0]

    # Traducir las etiquetas del modelo al espanol
    etiquetas = {"POS": "Positivo", "NEG": "Negativo", "NEU": "Neutral"}

    etiqueta = etiquetas.get(resultado["label"], resultado["label"])
    return {etiqueta: float(resultado["score"])}


# 3. INTERFAZ GRADIO
demo = gr.Interface(
    fn=analizar_sentimiento,
    inputs=gr.Textbox(
        label="Transcripción de audio",
        placeholder="Pegá acá la transcripción del audio...",
        lines=5,
    ),
    outputs=gr.Label(num_top_classes=3, label="Sentimiento detectado"),
    title="Analizador de sentimientos para habla",
    description=(
        "Analiza el sentimiento de transcripciones de audio con RoBERTuito, "
        "un modelo entrenado con tuits en español."
    ),
    examples=[
        ["La charla estuvo excelente, aprendí un montón."],
        ["Qué garrón de entrevista, no me fue nada bien."],
    ],
)

# 4. LANZAR LA APLICACION
if __name__ == "__main__":
    demo.launch()
```

### El archivo requirements.txt

```txt
transformers>=4.48,<5.0
torch>=2.6
```

`gradio` **no** se lista: el Space instala la versión que indica `sdk_version` en el `README.md`. Y se escribe a mano, con lo mínimo: no uses `uv pip freeze > requirements.txt`, que copia todos los paquetes con las versiones exactas de tu computadora y el build del Space puede fallar.

### El archivo README.md

El bloque entre `---` configura el Space. `sdk_version` tiene que ser la versión de Gradio que instalaste (la ves con `uv pip show gradio`).

```markdown
---
title: Analizador de Sentimientos para Habla
emoji: 🎙️
colorFrom: green
colorTo: blue
sdk: gradio
sdk_version: 6.17.3
python_version: "3.12"
app_file: app.py
pinned: false
license: mit
---

# Analizador de Sentimientos para Transcripciones de Audio

Analiza el sentimiento de transcripciones con RoBERTuito.
```

`license` es la licencia de **tu** código. Los modelos tienen la suya: revisá la ficha de cada uno antes de reutilizarlo.

## Parte 3: De notebook a script de Python

### Diferencias fundamentales

| Aspecto | Jupyter Notebook | Script de Python |
|---|---|---|
| **Ejecución** | Celda por celda | Todo de una vez |
| **Estado** | Interactivo, mutable | Declarativo, reproducible |
| **Imports** | Pueden estar dispersos | Al inicio del archivo |
| **Salidas** | Visuales, automáticas | Tienen que ser explícitas |
| **Depuración** | Fácil, paso a paso | Requiere logs o `print` |
| **Producción** | No recomendado | Ideal |

### Proceso de conversión: ejemplo del cuaderno 01

**ANTES: notebook (celdas separadas):**

In [ ]:
# Celda 1: instalacion (NO VA EN EL SCRIPT; va en requirements.txt)
# uv pip install transformers torch

# Celda 2: imports dispersos
from transformers import pipeline

# Celda 3: cargar modelo
clasificador = pipeline(
    "text-classification",
    model="pysentimiento/robertuito-sentiment-analysis",
)

# Celda 4: prueba interactiva (NO VA EN EL SCRIPT)
texto = "La charla estuvo excelente"
resultado = clasificador(texto)
print(resultado)

**DESPUÉS: script de Python (`app.py`):**

```python
"""
app.py - Analizador de sentimientos.
Convertido desde el cuaderno 01_explorar_tareas.ipynb
"""

# 1. IMPORTS: todos al inicio
import gradio as gr
from transformers import pipeline

# 2. CONFIGURACION: constantes globales
MODEL_NAME = "pysentimiento/robertuito-sentiment-analysis"
ETIQUETAS = {"POS": "Positivo", "NEG": "Negativo", "NEU": "Neutral"}

# 3. CARGA DEL MODELO: una sola vez
print("Cargando modelo...")
clasificador = pipeline("text-classification", model=MODEL_NAME)
print("Modelo cargado")


# 4. FUNCION DE PROCESAMIENTO: logica principal
def analizar_sentimiento(texto: str) -> dict:
    """Analiza el sentimiento y devuelve el resultado formateado.

    Args:
        texto: Texto a analizar.

    Returns:
        Diccionario {etiqueta: puntaje}, el formato que espera gr.Label.
    """
    if not texto.strip():
        raise gr.Error("Escribi un texto para analizar.")

    resultado = clasificador(texto)[0]
    etiqueta = ETIQUETAS.get(resultado["label"], resultado["label"])
    return {etiqueta: float(resultado["score"])}


# 5. INTERFAZ GRADIO: definicion de la UI
demo = gr.Interface(
    fn=analizar_sentimiento,
    inputs=gr.Textbox(label="Texto", lines=3),
    outputs=gr.Label(num_top_classes=3),
    title="Analizador de sentimientos",
)

# 6. PUNTO DE ENTRADA
if __name__ == "__main__":
    demo.launch()
```

### Checklist de conversión

- Sacar `pip install` o `uv pip install` del código y pasarlo a `requirements.txt`.
- Todos los imports al inicio del archivo.
- Cargar el modelo UNA vez (fuera de las funciones).
- Funciones con docstrings claros.
- Avisarle a la persona los errores con `gr.Error`, en lugar de devolver un texto de error donde se espera un puntaje.
- La interfaz de Gradio reemplaza los inputs interactivos.
- `if __name__ == "__main__"` como punto de entrada.

## Parte 4: Vibe engineering: desarrollo con IA generativa

### Qué es vibe engineering

**Vibe engineering** es una metodología de desarrollo que aprovecha los LLM como asistentes de programación. A diferencia del desarrollo tradicional, priorizás:

- **Iteración rápida** sobre planificación exhaustiva.
- **Conversación natural** con el LLM como compañero de trabajo.
- **Experimentación** sobre documentación detallada previa.
- **Prototipado rápido** para validar ideas.

**Comparación:**

```
Desarrollo tradicional:
  Diseño → Documentación → Implementación → Testing
  (días o semanas)

Vibe engineering:
  Idea → Pedido al LLM → Código → Probar → Iterar
  (minutos u horas)
```

### De un chat a un agente de código

Hay dos formas de trabajar con IA. En un **chat** copiás código, lo pegás y te devuelven texto. Un **agente de código** (como Claude Code o Codex) trabaja dentro de tu carpeta: lee los archivos del proyecto, los edita, ejecuta comandos y te muestra los cambios para que los apruebes. Los dos sirven para este laboratorio, y con los dos vale lo mismo: el agente escribe rápido y se equivoca con la misma seguridad, así que el trabajo es pedir bien y revisar.

- Leé el cambio completo antes de aceptarlo: ¿tocó solo lo que pediste?
- Ejecutá la app y tus checkpoints después de cada cambio.
- Nunca pegues claves, tokens ni datos personales en un pedido ni en el código.

### El ciclo iterativo MVP

El desarrollo con IA generativa sigue este ciclo:

```
1. MVP o feature
   ↓
2. Test
   ↓
3. Error
   ↓
4. Debug
   ↓
5. Checkpoint
   ↓
   ← Volver al inicio con una nueva feature
```

**Características del ciclo:**

- **Rápido**: cada iteración toma minutos, no horas.
- **Incremental**: construís sobre lo que ya funciona.
- **Guiado por checkpoints**: validás cada paso antes de avanzar.
- **Colaborativo**: la IA es tu par de programación.

### Ciclo de desarrollo con IA generativa (6 pasos)

**Paso 1: empezá con una idea.** Definí qué querés construir de forma simple:

```
"Quiero analizar el sentimiento de transcripciones de audio"
```

**Paso 2: construí un prototipo rápido con Python y Gradio.** Pedile a la IA el código base:

```
Prompt: "Creame una app Gradio que analice sentimientos de texto
usando el modelo pysentimiento/robertuito-sentiment-analysis"
```

**Paso 3: probá el prototipo.** Ejecutalo localmente y probalo con casos reales:

```bash
python app.py
# Probá con varios textos
```

**Paso 4: compartí el prototipo y obtené retroalimentación.** Mostralo a compañeros, docentes o usuarios potenciales. Anotá qué funciona bien, qué falla y qué falta.

**Paso 5: mejorá el prompt, el código o la interfaz según lo aprendido.** Iterá con la IA:

```
Prompt: "El modelo funciona pero necesito agregar análisis de emociones.
¿Qué modelo me recomendás y cómo lo integro sin romper lo que ya tengo?"
```

**Paso 6: decidí si vale la pena convertirlo en una aplicación completa.** Evaluá si el prototipo resuelve el problema, si hay interés real y si vale la pena pulirlo y desplegarlo. Si es así, volvé al paso 2 con nuevas features.

### Principios para llegar al MVP

**1. Dale a la IA solo la información relevante al MVP**

```
Mal prompt:
"Quiero una app de análisis de sentimientos que use 5 modelos diferentes,
con base de datos, autenticación, gráficos, exportación a PDF..."

Buen prompt:
"Creame una app Gradio simple que analice sentimientos de texto usando
RoBERTuito. Solo entrada de texto y salida de clasificación."
```

**2. Empezá pequeño y construí hacia arriba**

```
Iteración 1: clasificación básica
Iteración 2: agregar ejemplos
Iteración 3: mejorar la interfaz
Iteración 4: agregar una segunda pestaña para análisis en lote
```

**3. Dá contexto fundacional y detalles importantes**

```
Buen prompt:
"Estoy trabajando en una app de análisis de habla. Ya tengo un clasificador
de sentimientos funcionando con RoBERTuito en app.py que carga el modelo
una sola vez al inicio.

Necesito agregar análisis de emociones. El código actual es:
[pegar código]

¿Cómo integro pysentimiento/robertuito-emotion-analysis manteniendo
la misma arquitectura?"
```

## Parte 5: Vibe coding: los 4 tipos de pensamiento

Cuando desarrollás código (con o sin IA), aplicás cuatro tipos de pensamiento complementarios.

### 1. Pensamiento lógico

**Pregunta clave:** ¿qué es el juego? En programación: ¿qué hace este código? ¿Cuál es el flujo básico?

```python
# Ejemplo: entender la logica
def analizar_sentimiento(texto):
    # Entrada: texto
    # Proceso: pasar por el modelo
    # Salida: clasificacion
    resultado = clasificador(texto)
    return procesar(resultado)
```

**Aplicación con IA:**
```
Prompt: "Explicame paso a paso qué hace esta función:
[pegar código]"
```

### 2. Pensamiento analítico

**Pregunta clave:** ¿cómo juego el juego? En programación: ¿por qué funciona así? ¿Qué decisiones de diseño se tomaron?

```python
# Ejemplo: analizar decisiones de diseno
clasificador = pipeline("text-classification")  # Por que pipeline?
# Respuesta: pipeline abstrae tokenizacion, preprocesamiento y posprocesamiento

# Por que cargar el modelo al inicio y no dentro de la funcion?
# Respuesta: para no recargarlo en cada prediccion (es muy lento)
```

**Aplicación con IA:**
```
Prompt: "¿Por qué este código carga el modelo fuera de la función?
¿Qué pasaría si lo cargo adentro?"
```

### 3. Pensamiento computacional

**Pregunta clave:** ¿cuáles son los patrones detrás del juego? En programación: ¿qué patrones y optimizaciones puedo aplicar?

```python
# Ejemplo: identificar patrones ineficientes

# INEFICIENTE: recarga el modelo cada vez
def procesar_lote(textos):
    resultados = []
    for texto in textos:
        modelo_temporal = pipeline("...")
        resultados.append(modelo_temporal(texto))
    return resultados


# EFICIENTE: el modelo se carga UNA vez
modelo = pipeline("...")


def procesar_lote(textos):
    resultados = []
    for texto in textos:
        resultados.append(modelo(texto))
    return resultados
```

**Aplicación con IA:**
```
Prompt: "Este código funciona pero es lento. ¿Qué patrones de optimización
puedo aplicar?
[pegar código]"
```

### 4. Pensamiento procedimental

**Pregunta clave:** ¿cómo sobresalgo en el juego? En programación: ¿cuál es el paso a paso exacto? ¿Cómo lo ejecuto bien?

```python
# Ejemplo: descomponer en pasos
def app_completa():
    # Paso 1: cargar el modelo
    modelo = cargar_modelo()

    # Paso 2: definir la funcion de procesamiento
    def procesar(texto):
        return modelo(texto)

    # Paso 3: crear la interfaz
    interfaz = crear_gradio(procesar)

    # Paso 4: lanzar
    interfaz.launch()
```

**Aplicación con IA:**
```
Prompt: "Dame un plan paso a paso para convertir este notebook a un Space.
Numerá cada paso y decime qué archivos crear."
```

### Pirámide de complejidad

Los cuatro tipos de pensamiento se construyen uno sobre otro:

```
                  Procedimental
                (cómo sobresalir)
               /                 \
        Computacional              \
      (patrones y optimización)     \
         /                           \
    Analítico                         \
  (por qué funciona así)               \
       /                                \
   Lógico ――――――――――――――――――――――――――――――
 (qué hace)
```

**En la práctica:**

1. Empezás con **lógico**: «¿qué hace este código?».
2. Avanzás a **analítico**: «¿por qué está estructurado así?».
3. Aplicás **computacional**: «¿cómo lo optimizo?».
4. Dominás con **procedimental**: «¿cuál es el mejor flujo de trabajo?».

## Parte 6: Depuración sistemática y contexto

### La depuración es aburrida, pero necesaria

El mejor debugging es **metódico y exhaustivo**. Es aburrido, pero podés hacerlo divertido.

```
Problema: la lámpara no funciona
   ↓
¿Está enchufada? → No → Enchufar la lámpara
   ↓ Sí
¿La bombilla está quemada? → Sí → Reemplazar la bombilla
   ↓ No
Reparar la lámpara
```

### Estrategia de depuración con IA

**Objetivos:**
- Entender cómo funciona tu app.
- Entender dónde está el error.

**Preguntas clave:**
- ¿Cómo llegás a la raíz del problema?
- ¿Cómo le contás a la IA qué está mal?

**Método paso a paso:**

```python
# 1. Checkpoint: el modelo carga correctamente
print("CHECKPOINT 1: cargando modelo...")
modelo = pipeline("text-classification")
print("Modelo cargado")

# 2. Checkpoint: la funcion procesa correctamente
print("CHECKPOINT 2: probando funcion...")
resultado = analizar("Esto es una prueba")
assert isinstance(resultado, dict), "Debe devolver un dict"
print("Funcion correcta")

# 3. Checkpoint: la interfaz se crea
print("CHECKPOINT 3: creando interfaz...")
demo = gr.Interface(...)
print("Interfaz creada")

# 4. Checkpoint: la app se lanza localmente
print("CHECKPOINT 4: lanzando app...")
demo.launch()
print("App lanzada en http://127.0.0.1:7860")
```

**Comunicar errores a la IA:**

```
Mal prompt:
"No funciona, ayuda"

Buen prompt:
"Tengo este error en el CHECKPOINT 2:

Error:
TypeError: 'NoneType' object is not subscriptable
  File "app.py", line 15, in analizar_sentimiento
    etiqueta = resultado['label']

Código:
[pegar código relevante]

¿Qué está pasando?"
```

### Contexto: la ventana de contexto

**Contexto**, en el mundo de los LLM, es la cantidad de tokens que un modelo puede procesar en un momento dado. Esa cantidad se llama **ventana de contexto** y depende del modelo: hoy va de decenas de miles de tokens a cientos de miles o más. Consultá la documentación de la herramienta que uses.

**El contexto puede ser tu prompt, pero también:**
- El código que le mostraste antes.
- La conversación anterior.
- La documentación que le pasaste.
- Los errores anteriores.

**Estrategia de contexto efectivo:**

```
Contexto inicial (una vez):
"Estoy construyendo una app Gradio para análisis de sentimientos de
transcripciones de audio. Uso Python 3.12, transformers 4.x y el modelo
RoBERTuito. La app se va a publicar en Hugging Face Spaces.

La estructura es:
- app.py: código principal
- requirements.txt: dependencias
- README.md: metadatos

Arquitectura: cargamos el modelo UNA vez al inicio, luego definimos
las funciones de procesamiento y, por último, creamos la interfaz Gradio."

Luego, en cada prompt:
"[Tu pregunta específica, sin repetir todo el contexto]"
```

El LLM mantiene el contexto de la conversación, así que no tenés que repetirlo cada vez. Un agente de código además lee los archivos de tu carpeta: ahí el contexto es el propio proyecto.

## Parte 7: Desarrollo local con entorno virtual

### Flujo de trabajo completo

```bash
# 1. Crear la carpeta del proyecto
mkdir analizador-sentimientos
cd analizador-sentimientos

# 2. Crear el entorno virtual
uv venv --python 3.12

# 3. Activar el entorno (macOS o Linux)
source .venv/bin/activate

# 3. Activar el entorno (Windows, PowerShell)
.venv\Scripts\activate

# 4. Verificar que estás en el entorno: el prompt cambia

# 5. Instalar dependencias (en un solo comando)
uv pip install "transformers>=4.48,<5.0" "torch>=2.6" "gradio>=5.0"

# 6. Crear los archivos del proyecto
touch app.py requirements.txt README.md .gitignore
# En PowerShell: New-Item app.py, requirements.txt, README.md, .gitignore

# 7. Escribir requirements.txt a mano (ver la Parte 2)

# 8. Probar la app localmente
python app.py
# → Abre http://127.0.0.1:7860

# 9. Cuando termines de trabajar
deactivate
```

### Estructura del proyecto

```
analizador-sentimientos/
├── .venv/                 # ← Entorno virtual (NO subir a Git)
├── app.py                 # ← Código principal
├── requirements.txt       # ← Dependencias
├── README.md              # ← Metadatos y documentación
└── .gitignore             # ← Ignorar .venv/
```

## Parte 8: Despliegue a Hugging Face Spaces

> **Precaución.** Publicar deja tu aplicación al alcance de cualquiera: no subas datos personales, claves ni tokens. Esta parte requiere una cuenta de Hugging Face y no se probó en esta versión del laboratorio.

### Preparación

Antes de desplegar, verificá:

```bash
# Probar localmente
python app.py
# Funciona sin errores

# Verificar los archivos
ls
# app.py, requirements.txt y README.md existen
```

### Crear el Space en Hugging Face

1. Ir a <https://huggingface.co/new-space>.
2. Elegir un nombre, por ejemplo `analizador-sentimientos-habla`.
3. Elegir el SDK: **Gradio**.
4. Visibilidad: **Public** (o **Private** mientras probás).
5. Click en **Create Space**.

### Conectar Git y subir los archivos

```bash
# Inicializar el repositorio (si no lo hiciste)
git init

# Agregar los archivos
git add app.py requirements.txt README.md .gitignore

# Commit inicial
git commit -m "feat: analizador de sentimientos inicial"

# Conectar con Hugging Face
git remote add origin https://huggingface.co/spaces/TU_USUARIO/analizador-sentimientos-habla

# Configurar la rama principal
git branch -M main

# Subir
git push -u origin main
```

Al subir, Hugging Face te va a pedir autenticarte: usá un token de acceso con permiso de escritura o una clave SSH, nunca tu contraseña, y nunca guardes el token en el código.

### Verificar el despliegue

1. Ir a tu Space: `https://huggingface.co/spaces/TU_USUARIO/analizador-sentimientos-habla`.
2. Mirar los logs en la pestaña **Logs**.
3. Esperar a que termine el build (con `torch` puede tardar varios minutos).
4. Probar la aplicación.

### Actualizaciones posteriores

```bash
# Modificar app.py
# ...

# Commit y push
git add app.py
git commit -m "fix: mejorar el manejo de errores"
git push

# → El Space se actualiza automáticamente
```

## Parte 9: Ejercicio práctico guiado

### Qué se construye

Crear un Space que analice el sentimiento de transcripciones de audio con el modelo **RoBERTuito**, aplicando vibe engineering.

### Paso a paso

**Paso 1: crear el entorno virtual**

```bash
mkdir analizador-sentimientos-habla
cd analizador-sentimientos-habla
uv venv --python 3.12
source .venv/bin/activate    # Windows: .venv\Scripts\activate
```

**Paso 2: instalar dependencias**

```bash
uv pip install "transformers>=4.48,<5.0" "torch>=2.6" "gradio>=5.0"
```

Y escribí a mano un `requirements.txt` con `transformers>=4.48,<5.0` y `torch>=2.6` (ver la Parte 2).

**Paso 3: usar una IA para generar `app.py`**

Prompt al LLM o al agente de código (Claude, ChatGPT, etc.):

```
Creame un archivo app.py para una aplicación Gradio que:
1. Analice sentimientos de texto usando pysentimiento/robertuito-sentiment-analysis
2. Cargue el modelo UNA vez al inicio
3. Tenga una función que reciba texto y devuelva el sentimiento en español
4. Use gr.Interface con entrada de texto y salida gr.Label
5. Incluya 2 ejemplos en español rioplatense

El código debe ser simple, bien comentado y seguir buenas prácticas.
```

**Paso 4: crear `README.md`**

```markdown
---
title: Analizador de Sentimientos para Habla
emoji: 🎙️
colorFrom: green
colorTo: blue
sdk: gradio
sdk_version: 6.17.3
python_version: "3.12"
app_file: app.py
pinned: false
license: mit
---

# Analizador de Sentimientos para Transcripciones

Analiza el sentimiento de transcripciones de audio con RoBERTuito.
Entrenado con tuits en español.

## Uso
1. Pegá la transcripción del audio.
2. Click en Submit.
3. Mirá el resultado.
```

**Paso 5: crear `.gitignore`**

```
.venv/
venv/
env/
__pycache__/
*.pyc
.vscode/
.idea/
.DS_Store
```

**Paso 6: CHECKPOINT: probar localmente**

```bash
python app.py
# Abrir http://127.0.0.1:7860
# Probar con los textos de ejemplo
# Verificar que funciona
```

**Paso 7: desplegar a Hugging Face**

```bash
git init
git add .
git commit -m "feat: analizador de sentimientos inicial"
git remote add origin https://huggingface.co/spaces/TU_USUARIO/analizador-sentimientos-habla
git branch -M main
git push -u origin main
```

**Paso 8: CHECKPOINT: verificar el despliegue**

1. Ver los logs en Hugging Face.
2. Esperar el build.
3. Probar la app en la URL pública.
4. Si hay errores, copiar el error completo y pedirle ayuda a la IA.

## Parte 10: Ejercicios complementarios

### Ejercicio 1: iteración, agregar más ejemplos

**Prompt sugerido a la IA:**
```
Mi app.py actual tiene 2 ejemplos. Quiero agregar 3 ejemplos más de:
- Una entrevista laboral positiva
- Una reseña de podcast negativa
- Una transcripción de llamada neutral

Todos en español rioplatense coloquial.
```

### Ejercicio 2: nueva feature, análisis en lote

**Prompt sugerido a la IA:**
```
Tengo esta app de análisis de sentimientos. Quiero agregar una segunda
pestaña (Tab) donde pueda pegar múltiples transcripciones (una por línea)
y obtener el análisis de todas a la vez.

Código actual:
[pegar app.py]

¿Cómo lo hago usando gr.Blocks con gr.Tab?
```

### Ejercicio 3: comparación de modelos

**Prompt sugerido a la IA:**
```
Quiero crear una interfaz que compare RoBERTuito con otro modelo de
sentimiento lado a lado. El usuario ingresa un texto y ve los resultados
de ambos modelos.

Necesito:
1. Cargar ambos modelos al inicio
2. Una función que procese con ambos
3. Mostrar los resultados en dos columnas

¿Cómo estructuro el código?
```

BETO base no sirve para esta comparación: no trae cabeza de sentimiento. Buscá un modelo ya ajustado para sentimiento (por ejemplo, `nlptown/bert-base-multilingual-uncased-sentiment`, que devuelve estrellas y no `POS`, `NEG` y `NEU`) y revisá su ficha y su licencia.

## Recursos adicionales

### Documentación

- [Gradio](https://www.gradio.app/docs)
- [Hugging Face Spaces](https://huggingface.co/docs/hub/spaces)
- [Pipelines de Transformers](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [Vibe engineering, de Simon Willison](https://simonwillison.net/2025/Oct/7/vibe-engineering/)

### Modelos recomendados para habla y transcripciones

- **Sentimiento**: `pysentimiento/robertuito-sentiment-analysis`
- **Emociones**: `pysentimiento/robertuito-emotion-analysis`
- **Entidades**: `mrm8488/bert-spanish-cased-finetuned-ner`
- **Transcripción**: `openai/whisper-large-v3`

Antes de reutilizar un modelo, revisá su ficha y su licencia.

### Herramientas

- **VS Code**: editor recomendado.
- **uv**: entornos y paquetes de Python.
- **Un asistente de IA**: un chat (Claude, ChatGPT) o un agente de código.
- **Git**: control de versiones.

## Checklist final

### Antes de desplegar

- Entorno virtual creado con `uv` y activado.
- `app.py` funciona localmente.
- `requirements.txt` completo y escrito a mano.
- `README.md` con los metadatos YAML (`sdk_version` igual a tu versión de Gradio).
- `.gitignore` excluye `.venv/`.
- Probado con varios textos de ejemplo.

### Despliegue

- Space creado en Hugging Face.
- Git configurado y conectado.
- Push realizado sin errores.
- Build exitoso (revisar los logs).
- Aplicación funcional en la URL pública.

### Después del despliegue

- Probar todas las funcionalidades.
- Verificar los ejemplos.
- Compartir con compañeros.
- (Opcional) Conectar con GitHub para CI/CD.

## Qué te llevás

- Un proyecto reproducible: entorno con `uv`, dependencias escritas a mano y un `app.py` con el modelo cargado una sola vez.
- Con IA se trabaja en ciclos cortos: un pedido chico y bien dado, una prueba, un checkpoint. Quien desarrolla revisa lo que la IA escribe.
- Un Space publicado es una decisión: quién lo usa, qué datos escribe y quién lo mantiene.

**Para seguir:** integrar transcripción automática con Whisper, agregar visualizaciones de resultados y pensar qué cambiaría para un análisis en tiempo real.